Pydantic

In [1]:
import os 
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
model = init_chat_model("gpt-4.1-mini")

In [2]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The rating of the movie out of 10")

In [3]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure


_ChatModelBinding(bound=ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.0', 'langchain': '1.3.16', 'langchain-openai': '1.6.0'}}, profile={'name': 'GPT-4.1 mini', 'release_date': '2025-04-14', 'last_updated': '2025-04-14', 'open_weights': False, 'max_input_tokens': 1047576, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x000001D21CD66BD0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001D21D76F5D0>, root_client=<openai.OpenAI ob

In [5]:
response = model.invoke("provide me the details about movie Inception")
print(response.content)

Certainly! Here are the details about the movie **Inception**:

**Title:** Inception  
**Director:** Christopher Nolan  
**Writers:** Christopher Nolan  
**Producers:** Emma Thomas, Christopher Nolan, Charles Roven  
**Starring:**  
- Leonardo DiCaprio as Dom Cobb  
- Joseph Gordon-Levitt as Arthur  
- Ellen Page as Ariadne  
- Tom Hardy as Eames  
- Ken Watanabe as Saito  
- Cillian Murphy as Robert Fischer  
- Marion Cotillard as Mal Cobb  
- Michael Caine as Miles  

**Genre:** Science Fiction, Action, Thriller  
**Release Date:** July 16, 2010 (USA)  
**Runtime:** Approximately 148 minutes  
**Language:** English  

**Plot Summary:**  
Inception is a mind-bending thriller that explores the concept of entering and manipulating dreams. Dom Cobb (Leonardo DiCaprio) is a skilled thief who specializes in the art of "extraction," stealing secrets from within the subconscious during the dream state. Cobb is offered a chance to have his criminal record erased as payment for a seemingly imp

In [7]:
response = model_with_structure.invoke("provide details about movie Inception")
print(response)

title='Inception' year=2010 director='Christopher Nolan' rating=8.8


In [8]:
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

In [9]:
##message output along side parsed output

In [10]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A moview with details"""
    title:str = Field(...,description="The title of the movie")
    year:int = Field(...,description="The year the movie was released")
    director:str = Field(...,description="The director of the movie")
    rating:float = Field(...,description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie,include_raw=True)

response = model_with_structure.invoke("Provide the details about the movie Titanic")
response


{'raw': AIMessage(content='{"title":"Titanic","year":1997,"director":"James Cameron","rating":7.8}', additional_kwargs={'parsed': Movie(title='Titanic', year=1997, director='James Cameron', rating=7.8), 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 127, 'total_tokens': 149, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_b0bb179f09', 'id': 'chatcmpl-EKWEVKBIHeY1BBITBPr7U54LOenK9', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a06e6f-1b18-7462-9c47-383ae9a10a68-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 127, 'output_tokens': 22, 't

In [11]:
##Nested Structure

In [12]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name:str
    role:str

class MovieDetails(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    genres: list[str]
    budget:float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)
response = model_with_structure.invoke("Provide details about the movie Inception")
response


MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames')], genres=['Action', 'Adventure', 'Sci-Fi'], budget=160.0)

TypeDict

It provides a simpler alternativ using Python's built in typing, ideal when you don't need runtime validation

In [13]:
from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    """A moview with details"""
    title:Annotated[str,...,"The title of the movie"]
    year:Annotated[int,...,"The year the movie was released"]
    director:Annotated[str,...,"The director of the movie"]
    rating:Annotated[float,...,"The movie's rating out of 10"]

model_withtypedict = model.with_structured_output(MovieDict)
response = model_withtypedict.invoke("please provide me the details of the movie avengers")
response

{'title': 'The Avengers',
 'year': 2012,
 'director': 'Joss Whedon',
 'rating': 8.0}

In [14]:
class Actor(TypedDict):
    name:str
    role:str

class MovieDetails(TypedDict):
    title:str
    year:int
    cast:list[Actor]
    genres: list[str]
    budget:float | None = Field(None, description="Budget in millions USD")

model_withtypedict = model.with_structured_output(MovieDetails)
response = model_withtypedict.invoke("please provide me the details of the movie avengers")
response

{'title': 'The Avengers',
 'year': 2012,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'},
  {'name': 'Tom Hiddleston', 'role': 'Loki'},
  {'name': 'Samuel L. Jackson', 'role': 'Nick Fury'}],
 'genres': ['Action', 'Adventure', 'Sci-Fi'],
 'budget': 220000000}

In [16]:
model.profile

{'name': 'GPT-4.1 mini',
 'release_date': '2025-04-14',
 'last_updated': '2025-04-14',
 'open_weights': False,
 'max_input_tokens': 1047576,
 'max_output_tokens': 32768,
 'text_inputs': True,
 'image_inputs': True,
 'audio_inputs': False,
 'pdf_inputs': True,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': False,
 'tool_calling': True,
 'structured_output': True,
 'attachment': True,
 'temperature': True,
 'image_url_inputs': True,
 'pdf_tool_message': True,
 'image_tool_message': True,
 'tool_choice': True,
 'tool_call_streaming': True}

Data Classes

A data class is a class typically contatining mainly data, although there aren't really any restrictions. You create it using the @dataclass decorator

In [18]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information for the person"""
    name:str = Field(description="The name of the person")
    email:str=Field(description="The email address of the person")
    phone:str = Field(description="The phone number of the person")

agent = create_agent(
    model="gpt-5.4-mini",
    response_format=ContactInfo
)

result = agent.invoke({
    "messages":[{"role":"user", "content":"Extract contact info from: John Doe, john@example.com, (555)123-4567"}]              
})

print(result["structured_response"])

name='John Doe' email='john@example.com' phone='(555)123-4567'


In [19]:
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person"""
    name: str 
    email:str
    phone:str

agent = create_agent(
    model="gpt-5.4-mini",
    response_format=ContactInfo
)

result = agent.invoke({
    "messages":[{"role":"user", "content":"Extract contact info from: John Doe, john@example.com, (555)123-4567"}]              
})

print(result["structured_response"])



ContactInfo(name='John Doe', email='john@example.com', phone='(555)123-4567')


In [20]:
result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555)123-4567', additional_kwargs={}, response_metadata={}, id='d6e0d056-782e-493a-b611-34e32268231e'),
  AIMessage(content='{"name":"John Doe","email":"john@example.com","phone":"(555)123-4567"}', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 177, 'total_tokens': 206, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EKWgDdTbwXP2SM2Cwmt6i17JAVbnZ', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a06e89-5143-7641-af19-914eb